## Setup

This notebook provides both automatic (`auto`) configuration and full manual overrides for running, converting, quantizing, and benchmarking AI models. Leave parameters at `auto` or defaults for zero-config execution, or customize any field below.

### 1. Code & Workspace Location
- `code_source`: Source used to load `workspace_core` (`github`, `google_drive`, `gdrive_relative`, `url`, `local_dir`).
- `code_path`: Repository URL, Drive folder, archive URL, or local directory path.
- `code_branch`: Git branch or tag to clone when `code_source` is `github`.
- `force_update_code`: Re-pull or re-sync `workspace_core` even if `/content/app` already exists.
- `workspace_mode`: Storage strategy (`local_ssd` for Drive-free Colab execution, `hybrid`, `google_drive`, `gdrive_relative`, `custom_path`).
- `workspace_path`: Folder relative to `MyDrive` (`hybrid` / `gdrive_relative`), absolute path (`google_drive` / `custom_path`), or ignored in `local_ssd`.
- `env_isolation_mode`: Package isolation (`auto`, `isolated_venv`, `system`).
- `auto_install_dependencies`: Automatically fetch missing engine binaries or Python dependencies at runtime.

### 2. Input Source & Location
- `input_type`: Input handler (`auto`, `huggingface`, `direct_upload`, `local_path`, `google_drive`, `gdrive_relative`, `gdrive_share`, `url`, `civitai`, `git_repo`).
- `input_path`: Repo ID, URL, share ID, or filesystem path (ignored when `input_type="direct_upload"`).
- `input_file`: Specific filename or subpath inside a repo, folder, or archive (blank = auto-select).
- `hf_revision`: Optional Hugging Face branch, tag, or commit hash.
- `unpack_archive`: Automatically extract `.zip` or `.tar.*` input archives.
- `model_type_override`: Force model family (`auto`, `llm-gguf`, `vlm-gguf`, `embedding-gguf`, `llm-safetensors`, `diffusion`, `audio`).

### 3. Task, Auto Strategy & Engine
- `task`: Workflow mode (`serve`, `convert_and_quantize`, `convert_and_serve`, `benchmark`, `inspect_only`, `dashboard_only`, `serve_and_dashboard`).
- `selection_strategy`: Auto-Selector policy (`balanced`, `max_quality`, `max_speed`, `max_context`, `low_vram_safe`).
- `engine`: Execution engine (`auto` or explicit backend).
- `preset_name`: Base `ServerPreset` from `configs/presets.yaml` (`auto` or explicit preset key).
- `quantizer`: Conversion/quantization backend (`auto`, `none`, or explicit converter).
- `target_quant`: Desired quantization (`auto` or explicit tag such as `Q4_K_M`, `IQ4_KS`, `Q4_K_XL`, `4.0`, `AWQ-4bit`, `GPTQ-4bit`).

### 4. Server Tuning: Auto & Manual
- `context_size`: Context window (`auto` or exact integer token limit).
- `gpu_offload_mode`: GPU layer strategy (`auto`, `all`, `cpu`, `manual`).
- `n_gpu_layers`: Explicit GPU layers when `gpu_offload_mode="manual"` (or `-1` for all, `0` for CPU).
- `flash_attn_mode`: Flash Attention (`auto`, `on`, `off`).
- `cache_type_k` / `cache_type_v`: KV cache precision (`auto`, `q8_0`, `q4_0`, `f16`).
- `parallel`, `batch_size`, `ubatch_size`, `threads`, `split_mode`, `tensor_split`, `override_tensor`, `quantization_mode`, `dtype`, `gpu_memory_utilization`, `rope_scaling`, `chat_template`, `reasoning_format`, `mlock`, `mmap`, `mmproj_file`, `custom_server_args`: Fine-grained manual engine controls.

### 5. Conversion, Output & Networking
- Conversion options (`outtype`, `use_imatrix`, `imatrix_chunks`, `split_max_size`, `merge_lora`, `base_model_name`, `gptq_bits`, `gptq_group_size`, `exl2_bpw`, `cleanup_intermediate`).
- Output options (`output_type`, `output_path`, `output_name`, `zip_output`, `hf_private`, `cleanup_after_export`).
- Networking & Secrets (`enable_tunnel`, `tunnel_type`, `tunnel_token`, `tunnel_hostname`, `enable_api_auth`, `api_key_override`, `port`, `dashboard_port`, `hf_token`, `civitai_token`).

After configuring the form below, run **Runtime > Run All** (`Ctrl + F9`) or run individual sections below.

In [ ]:
#@title Setup Configuration
#@markdown ---
#@markdown ### 1. Code & Workspace Location
code_source = "github" #@param ["github", "google_drive", "gdrive_relative", "url", "local_dir"]
code_path = "https://github.com/Neutron84/universal-ai-workspace.git" #@param {type:"string"}
code_branch = "main" #@param {type:"string"}
force_update_code = False #@param {type:"boolean"}
workspace_mode = "local_ssd" #@param ["local_ssd", "hybrid", "google_drive", "gdrive_relative", "custom_path"]
workspace_path = "AI_Workspace" #@param {type:"string"}
env_isolation_mode = "auto" #@param ["auto", "isolated_venv", "system"]
auto_install_dependencies = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 2. Input Source & Location
input_type = "auto" #@param ["auto", "huggingface", "direct_upload", "local_path", "google_drive", "gdrive_relative", "gdrive_share", "url", "civitai", "git_repo"]
input_path = "Qwen/Qwen2.5-0.5B-Instruct-GGUF" #@param {type:"string"}
input_file = "" #@param {type:"string"}
hf_revision = "" #@param {type:"string"}
unpack_archive = True #@param {type:"boolean"}
model_type_override = "auto" #@param ["auto", "llm-gguf", "vlm-gguf", "embedding-gguf", "llm-safetensors", "diffusion", "audio"]

#@markdown ---
#@markdown ### 3. Task, Auto Strategy & Engine
task = "serve" #@param ["serve", "convert_and_quantize", "convert_and_serve", "benchmark", "inspect_only", "dashboard_only", "serve_and_dashboard"]
selection_strategy = "balanced" #@param ["balanced", "max_quality", "max_speed", "max_context", "low_vram_safe"]
engine = "auto" #@param ["auto", "llama.cpp", "ik_llama", "vllm", "transformers", "unsloth", "exllamav2", "diffusers", "faster-whisper"]
preset_name = "auto" #@param ["auto", "gguf_balanced", "gguf_long_ctx", "ik_llama_moe", "unsloth_fast", "transformers_4bit", "vllm_high_throughput", "exllamav2_fast", "vlm_multimodal", "diffusers_sd", "whisper_audio"]
quantizer = "auto" #@param ["auto", "none", "llama.cpp", "ik_llama", "unsloth", "gptqmodel", "autoawq", "exllamav2", "ct2"]
target_quant = "auto" #@param ["auto", "Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q4_K_XL", "IQ4_XS", "IQ4_K", "IQ4_KS", "IQ3_K", "IQ3_KS", "IQ2_K", "GPTQ-4bit", "AWQ-4bit", "FP8", "4.0", "F16"] {allow-input: true}

#@markdown ---
#@markdown ### 4. Server Tuning (Auto & Full Manual)
context_size = "4096" #@param ["auto", "2048", "4096", "8192", "16384", "32768", "65536"] {allow-input: true}
gpu_offload_mode = "auto" #@param ["auto", "all", "cpu", "manual"]
n_gpu_layers = -1 #@param {type:"integer"}
flash_attn_mode = "auto" #@param ["auto", "on", "off"]
cache_type_k = "auto" #@param ["auto", "q8_0", "q4_0", "f16"]
cache_type_v = "auto" #@param ["auto", "q8_0", "q4_0", "f16"]
parallel = 1 #@param {type:"integer"}
batch_size = 512 #@param {type:"integer"}
ubatch_size = 512 #@param {type:"integer"}
threads = 0 #@param {type:"integer"}
split_mode = "layer" #@param ["layer", "row", "none"]
tensor_split = "" #@param {type:"string"}
override_tensor = "" #@param {type:"string"}
quantization_mode = "auto" #@param ["auto", "bnb_4bit", "bnb_8bit", "fp16", "bf16", "fp32", "fp8", "awq", "gptq"]
dtype = "auto" #@param ["auto", "fp16", "bf16", "fp32"]
gpu_memory_utilization = 0.90 #@param {type:"number"}
rope_scaling = "" #@param ["", "none", "linear", "yarn"] {allow-input: true}
chat_template = "" #@param {type:"string"}
reasoning_format = "" #@param ["", "none", "deepseek"] {allow-input: true}
mlock = False #@param {type:"boolean"}
mmap = True #@param {type:"boolean"}
mmproj_file = "" #@param {type:"string"}
custom_server_args = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 5. Conversion & Quantization Details
outtype = "f16" #@param ["f16", "bf16", "f32", "q8_0"]
use_imatrix = False #@param {type:"boolean"}
imatrix_chunks = 100 #@param {type:"integer"}
split_max_size = "" #@param {type:"string"}
merge_lora = True #@param {type:"boolean"}
base_model_name = "" #@param {type:"string"}
gptq_bits = 4 #@param {type:"integer"}
gptq_group_size = 128 #@param {type:"integer"}
exl2_bpw = 4.0 #@param {type:"number"}
cleanup_intermediate = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 6. Output Target & Location
output_type = "none" #@param ["none", "local_ssd", "direct_download", "google_drive", "gdrive_relative", "gdrive_direct", "huggingface_hub", "custom_path"]
output_path = "AI_Workspace/outputs" #@param {type:"string"}
output_name = "" #@param {type:"string"}
zip_output = False #@param {type:"boolean"}
hf_private = False #@param {type:"boolean"}
cleanup_after_export = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### 7. Runtime, Tunnel & Secrets
enable_tunnel = True #@param {type:"boolean"}
tunnel_type = "cloudflared" #@param ["cloudflared", "ngrok", "none"]
tunnel_token = "" #@param {type:"string"}
tunnel_hostname = "" #@param {type:"string"}
enable_api_auth = True #@param {type:"boolean"}
api_key_override = "" #@param {type:"string"}
port = 8000 #@param {type:"integer"}
dashboard_port = 7860 #@param {type:"integer"}
hf_token = "" #@param {type:"string"}
civitai_token = "" #@param {type:"string"}

import os
import shutil
import subprocess
import sys
from pathlib import Path

# 1. Mount Google Drive only if required by any configured mode
drive_required_modes = {"google_drive", "gdrive_relative", "gdrive_direct", "hybrid"}
needs_drive = any(
    str(val).strip().lower() in drive_required_modes
    for val in (code_source, workspace_mode, input_type, output_type)
)

if needs_drive:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").is_dir():
            print("Mounting Google Drive to /content/drive...")
            drive.mount("/content/drive")
        else:
            print("Google Drive is already mounted.")
    except Exception as e:
        print(f"Note: Google Drive mount unavailable ({e}). Operating in local storage mode.")
else:
    print("Local storage mode active. Google Drive mount skipped.")

# 2. Acquire workspace source code into local environment
LOCAL_APP_DIR = Path("/content/app")
if (Path.cwd() / "workspace_core").is_dir() and not Path("/content").is_dir():
    LOCAL_APP_DIR = Path.cwd()
    print(f"Using local repository at {LOCAL_APP_DIR}.")
else:
    LOCAL_APP_DIR.mkdir(parents=True, exist_ok=True)
    has_code = (LOCAL_APP_DIR / "workspace_core").is_dir()

    if has_code and not force_update_code:
        print(f"Using existing workspace code at {LOCAL_APP_DIR}.")
    elif code_source == "github":
        repo_url = code_path if (code_path.startswith("http") or code_path.startswith("git")) else f"https://github.com/{code_path}.git"
        if (LOCAL_APP_DIR / ".git").is_dir():
            print(f"Updating repository in {LOCAL_APP_DIR}...")
            subprocess.run(["git", "-C", str(LOCAL_APP_DIR), "pull", "--ff-only"], check=False)
        else:
            if has_code:
                shutil.rmtree(str(LOCAL_APP_DIR))
                LOCAL_APP_DIR.mkdir(parents=True, exist_ok=True)
            print(f"Cloning repository {repo_url} (branch: {code_branch}) into {LOCAL_APP_DIR}...")
            clone_cmd = ["git", "clone", "--depth", "1"]
            if code_branch and code_branch.strip():
                clone_cmd.extend(["--branch", code_branch.strip()])
            clone_cmd.extend([repo_url, str(LOCAL_APP_DIR)])
            subprocess.run(clone_cmd, check=True)
    elif code_source in ["google_drive", "gdrive_relative"]:
        drive_src = Path(code_path) if code_path.startswith("/content/drive") else (Path("/content/drive/MyDrive") / code_path.strip().lstrip("/"))
        if drive_src.is_dir() and (drive_src / "workspace_core").is_dir():
            print(f"Syncing workspace code from Google Drive to {LOCAL_APP_DIR}...")
            for item in ["workspace_core", "configs", "pyproject.toml", "README.md"]:
                s = drive_src / item
                d = LOCAL_APP_DIR / item
                if s.is_dir():
                    shutil.copytree(str(s), str(d), dirs_exist_ok=True)
                elif s.is_file():
                    shutil.copy2(str(s), str(d))
    elif code_source == "url":
        print(f"Downloading source archive from {code_path}...")
        import urllib.request, zipfile
        tmp_zip = Path("/content/source_archive.zip")
        urllib.request.urlretrieve(code_path, str(tmp_zip))
        with zipfile.ZipFile(str(tmp_zip), "r") as z:
            z.extractall(str(LOCAL_APP_DIR))
    elif code_source == "local_dir":
        loc_src = Path(code_path).resolve()
        if loc_src != LOCAL_APP_DIR.resolve() and (loc_src / "workspace_core").is_dir():
            for item in ["workspace_core", "configs", "pyproject.toml"]:
                s = loc_src / item
                if s.is_dir():
                    shutil.copytree(str(s), str(LOCAL_APP_DIR / item), dirs_exist_ok=True)

# 3. Register python path & tokens
if str(LOCAL_APP_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_APP_DIR))

from workspace_core import get_config, SecurityManager
if hf_token.strip():
    SecurityManager.set_cached_secret("HF_TOKEN", hf_token.strip())
if civitai_token.strip():
    SecurityManager.set_cached_secret("CIVITAI_API_KEY", civitai_token.strip())

config = get_config(workspace_mode=workspace_mode, workspace_path=workspace_path)
config.update_settings(
    selection_strategy=selection_strategy,
    env_isolation_mode=env_isolation_mode,
    auto_install_dependencies=auto_install_dependencies,
    tunnel_hostname=tunnel_hostname.strip(),
)
print(f"Workspace initialized. Mode: {config.mode.value} | Strategy: {selection_strategy} | Root: {config.drive_root}")

In [ ]:
#@title 1. Environment & Hardware Diagnostics
#@markdown Installs system dependencies, core packages, engine binaries, and runs pre-flight diagnostics.
from workspace_core import EnvironmentManager, get_hardware_profile

env_mgr = EnvironmentManager(config)
env_mgr.ensure_system_packages()
env_mgr.install_profile("core", isolate=False)

if enable_tunnel and tunnel_type == "cloudflared":
    env_mgr.fetch_cloudflared()

target_engine = engine.lower().strip()
target_quantizer = quantizer.lower().strip()

if target_engine in ("ik_llama", "ik_llama.cpp") or target_quantizer in ("ik_llama", "ik_llama.cpp"):
    env_mgr.fetch_ik_llama_cpp()
elif target_engine in ("auto", "llama.cpp", "llama_cpp") or target_quantizer in ("auto", "llama.cpp", "llama_cpp"):
    env_mgr.fetch_llama_cpp()

if target_engine not in ("auto", "llama.cpp", "llama_cpp", "ik_llama", "ik_llama.cpp"):
    env_mgr.ensure_ready(target_engine)
if target_quantizer not in ("auto", "none", "llama.cpp", "llama_cpp", "ik_llama", "ik_llama.cpp"):
    env_mgr.ensure_ready(target_quantizer)

hw = get_hardware_profile()
gpu_desc = (
    f"{hw.primary_gpu.name} ({hw.total_vram_mb:.0f} MB VRAM, CC {hw.primary_gpu.compute_capability[0]}.{hw.primary_gpu.compute_capability[1]})"
    if hw.primary_gpu else "No GPU detected (CPU mode)"
)

print("System Diagnostics:")
print(f"  GPU:      {gpu_desc}")
print(f"  RAM:      {hw.ram_available_gb:.1f} GB available / {hw.ram_total_gb:.1f} GB total")
print(f"  SSD:      {hw.ssd_free_gb:.1f} GB free")
print(f"  CUDA:     {hw.cuda_version or 'N/A'}")
print(f"  Python:   {hw.python_version}")

In [ ]:
#@title 2. Input Resolution, Inspection & Planning
#@markdown Resolves input source URI, inspects model architecture, and computes Auto/Manual execution plan.
from workspace_core import ModelInspector, IOResolver, AutoSelector, SecurityManager

def _colab_upload_handler():
    from google.colab import files
    print("Upload your model file or archive:")
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        return str(Path("/content") / uploaded_name)
    raise ValueError("No file uploaded.")

resolved_input_path = input_path
if input_type == "direct_upload":
    resolved_input_path = _colab_upload_handler()

resolved_input = IOResolver.resolve_input(
    uri=resolved_input_path,
    mode=input_type,
    download_dir=config.runtime_root / "inputs",
    hf_token=SecurityManager.get_hf_token(),
    civitai_token=SecurityManager.get_civitai_api_key(),
    unpack_archive=unpack_archive,
    input_file=input_file or None,
    hf_revision=hf_revision or None,
)

if task != "dashboard_only":
    inspector = ModelInspector(hf_token=SecurityManager.get_hf_token())
    insp_target = str(resolved_input.local_path) if resolved_input.local_path else resolved_input_path
    insp = inspector.inspect(
        insp_target,
        input_mode=input_type,
        hf_revision=hf_revision or None,
        model_type_override=model_type_override if model_type_override != "auto" else None,
        input_file=input_file or None,
    )
    if insp.error:
        raise RuntimeError(f"Inspection error: {insp.error}")

    ctx_parsed = context_size if str(context_size).strip().lower() == "auto" else int(str(context_size).strip())
    fa_parsed = None if flash_attn_mode == "auto" else (flash_attn_mode == "on")
    ngl_override = n_gpu_layers if (gpu_offload_mode == "manual" or (gpu_offload_mode == "auto" and n_gpu_layers != -1)) else None

    selector = AutoSelector(config=config)
    rec = selector.recommend(
        inspection=insp,
        requested_ctx=ctx_parsed,
        override_quant=target_quant if target_quant != "auto" else None,
        override_engine=engine if engine != "auto" else None,
        override_quantizer=quantizer if quantizer != "auto" else None,
        task=task,
        override_n_gpu_layers=ngl_override,
        override_flash_attn=fa_parsed,
        override_cache_type_k=cache_type_k if cache_type_k != "auto" else None,
        override_cache_type_v=cache_type_v if cache_type_v != "auto" else None,
        override_gpu_offload_mode=gpu_offload_mode,
        override_parallel=parallel,
        override_quantization_mode=quantization_mode if quantization_mode != "auto" else None,
        override_mmproj_file=mmproj_file.strip() or None,
        strategy=selection_strategy,
    )
    print("Decision Summary:")
    print(f"  {rec.decision_summary}")
    print(f"  Selected Engine:    {rec.recommended_engine}")
    print(f"  Selected Quantizer: {rec.recommended_converter or 'none'}")
    print(f"  Recommended Quant:  {rec.recommended_quant}")
    print(f"  VRAM Budget:        {rec.vram_budget_mb:.0f} MB")
    print(f"  Context Window:     {rec.recommended_ctx} tokens")
    print(f"  GPU Offload Layers: {rec.recommended_n_gpu_layers}")
else:
    print("task='dashboard_only' selected; model inspection skipped.")

In [ ]:
#@title 3. Task Execution (Serve, Convert, Benchmark, or Dashboard)
#@markdown Executes the configured task using the unified WorkspaceOrchestrator.
from workspace_core import LaunchRequest, WorkspaceOrchestrator
from workspace_core.dashboard import launch_dashboard

orchestrator = WorkspaceOrchestrator(config)
ctx_parsed = context_size if str(context_size).strip().lower() == "auto" else int(str(context_size).strip())

launch_req = LaunchRequest(
    input_type=input_type,
    input_path=resolved_input_path,
    input_file=input_file.strip() or None,
    hf_revision=hf_revision.strip() or None,
    unpack_archive=unpack_archive,
    model_type_override=model_type_override if model_type_override != "auto" else None,
    task=task,
    strategy=selection_strategy,
    engine=engine,
    quantizer=quantizer,
    target_quant=target_quant,
    preset_name=preset_name if preset_name != "auto" else None,
    context_size=ctx_parsed,
    gpu_offload_mode=gpu_offload_mode,
    n_gpu_layers=n_gpu_layers,
    flash_attn=flash_attn_mode,
    cache_type_k=cache_type_k,
    cache_type_v=cache_type_v,
    parallel=parallel,
    batch_size=batch_size,
    ubatch_size=ubatch_size,
    threads=threads,
    split_mode=split_mode,
    tensor_split=tensor_split.strip() or None,
    override_tensor=override_tensor.strip() or None,
    mlock=mlock,
    mmap=mmap,
    rope_scaling=rope_scaling.strip() or None,
    chat_template=chat_template.strip() or None,
    reasoning_format=reasoning_format.strip() or None,
    quantization_mode=quantization_mode,
    dtype=dtype,
    gpu_memory_utilization=gpu_memory_utilization,
    mmproj_file=mmproj_file.strip() or None,
    custom_server_args=custom_server_args,
    outtype=outtype,
    use_imatrix=use_imatrix,
    imatrix_chunks=imatrix_chunks,
    split_max_size=split_max_size.strip() or None,
    merge_lora=merge_lora,
    base_model_name=base_model_name.strip() or None,
    gptq_bits=gptq_bits,
    gptq_group_size=gptq_group_size,
    exl2_bpw=exl2_bpw,
    cleanup_intermediate=cleanup_intermediate,
    output_type="none",  # Handled explicitly in Cell 4 below
    enable_tunnel=enable_tunnel,
    tunnel_type=tunnel_type,
    tunnel_token=tunnel_token.strip() or None,
    tunnel_hostname=tunnel_hostname.strip() or None,
    enable_api_auth=enable_api_auth,
    api_key_override=api_key_override.strip() or None,
    port=port,
    dashboard_port=dashboard_port,
    hf_token=hf_token.strip() or None,
    civitai_token=civitai_token.strip() or None,
    env_isolation_mode=env_isolation_mode,
    auto_install_dependencies=auto_install_dependencies,
)

if task == "dashboard_only":
    print(f"Launching management dashboard on port {dashboard_port}...")
    launch_dashboard(port=dashboard_port, share=True, config=config)
else:
    launch_res = orchestrator.run(launch_req)
    task_artifact_path = launch_res.artifact_path
    task_companion_files = launch_res.companion_files

    if launch_res.system_state:
        st = launch_res.system_state
        print("-" * 50)
        print(f"Server Status:   {'Running' if st.is_server_running else 'Failed'}")
        print(f"Backend Engine:  {st.engine_name}")
        print(f"Local Endpoint:  http://127.0.0.1:{st.port}/v1")
        if st.public_url:
            print(f"Public Tunnel:   {st.public_url}/v1")
        if st.api_key:
            print(f"API Key:         {st.api_key}")
        print("-" * 50)

    if launch_res.benchmark_results:
        print("Benchmark Results:", launch_res.benchmark_results)

    if task == "serve_and_dashboard":
        print(f"Launching management dashboard on port {dashboard_port}...")
        launch_dashboard(port=dashboard_port, share=True, config=config)

In [ ]:
#@title 4. Artifact Export & Output Management
#@markdown Exports generated models or shards to Local SSD, Google Drive, Hugging Face Hub, or browser download.
from workspace_core import OutputManager, ProcessManager, SecurityManager

if output_type != "none" and "task_artifact_path" in locals() and task_artifact_path and task_artifact_path.exists():
    out_mgr = OutputManager(paths=config.paths, hf_token=SecurityManager.get_hf_token())
    pm_check = ProcessManager(config)
    st_check = pm_check.get_state()
    protected = [task_artifact_path] if st_check.is_server_running else []

    print(f"Exporting artifact {task_artifact_path.name} to {output_type}...")
    export_res = out_mgr.export_artifact(
        source_path=task_artifact_path,
        output_mode=output_type,
        destination=output_path,
        output_name=output_name or None,
        zip_output=zip_output,
        cleanup_source=cleanup_after_export,
        companion_files=locals().get("task_companion_files", []),
        hf_private=hf_private,
        protected_paths=protected,
    )
    print(f"Export Status: {export_res.get('message', 'Complete')}")
    if export_res.get("exported_path"):
        print(f"Location:      {export_res.get('exported_path')}")
else:
    print("Output export skipped (output_type=none or no artifact to export).")

In [ ]:
#@title 5. Verification & Client Integration
#@markdown Verifies running server endpoints and prints ready-to-use Python and cURL client snippets.
import json
import requests
from workspace_core import ProcessManager

pm = ProcessManager(config)
state = pm.get_state()

if state.is_server_running:
    base_url = f"http://127.0.0.1:{state.port}"
    headers = {"Authorization": f"Bearer {state.api_key}"} if state.api_key else {}

    try:
        r = requests.get(f"{base_url}/v1/models", headers=headers, timeout=5)
        print("GET /v1/models response:")
        print(json.dumps(r.json(), indent=2))
    except Exception as e:
        print(f"Healthcheck error: {e}")

    if state.engine_name in ("diffusion", "diffusers"):
        print("\nDiffusion engine active. Ready to generate images via POST /v1/images/generations.")
    elif state.engine_name in ("audio", "faster-whisper"):
        print("\nAudio engine active. Ready to transcribe audio via POST /v1/audio/transcriptions.")
    else:
        print("\nTesting streaming chat completion:")
        payload = {
            "model": state.active_model or "default",
            "messages": [{"role": "user", "content": "Explain neural network weights in two sentences."}],
            "stream": True,
            "temperature": 0.2,
        }
        try:
            r = requests.post(f"{base_url}/v1/chat/completions", json=payload, headers=headers, stream=True, timeout=30)
            for line in r.iter_lines():
                if line:
                    dec = line.decode("utf-8")
                    if dec.startswith("data: "):
                        data_str = dec[6:].strip()
                        if data_str == "[DONE]":
                            break
                        chunk = json.loads(data_str)
                        delta = chunk.get("choices", [{}])[0].get("delta", {}).get("content", "")
                        print(delta, end="", flush=True)
            print("\n\nVerification completed successfully.")
        except Exception as e:
            print(f"Stream verification error: {e}")

    py_snippet, curl_snippet = ProcessManager.generate_code_snippets(state)
    print("\n" + "=" * 30 + " Python Client Example " + "=" * 30)
    print(py_snippet)
    print("=" * 30 + " cURL Client Example " + "=" * 30)
    print(curl_snippet)
else:
    print("Server is not currently running. Verification skipped.")

## Disclaimer

Google Colab provides access to shared computing resources subject to availability and usage limits. Free-tier GPU allocations are subject to timeout restrictions and preemption. Extended GPU usage or excessive memory consumption may result in session termination by the platform.

## License

This project is licensed under the MIT License. See the [LICENSE](LICENSE) file for details.